In [8]:
import boto3
import os
import pandas as pd
import io
import json
import duckdb

s3 = boto3.client("s3", endpoint_url=os.environ["MLFLOW_S3_ENDPOINT_URL"])

In [35]:
# Extract time_in_daylight from each file
s3 = boto3.client(
    "s3",
    endpoint_url=os.environ["MLFLOW_S3_ENDPOINT_URL"],
    aws_access_key_id=os.environ["AWS_ACCESS_KEY_ID"],
    aws_secret_access_key=os.environ["AWS_SECRET_ACCESS_KEY"],
)

# List all JSON files under metrics/json/
paginator = s3.get_paginator("list_objects_v2")
pages = paginator.paginate(Bucket="health-data", Prefix="metrics/json/amanda")
keys = [obj["Key"] for page in pages for obj in page.get("Contents", []) if obj["Key"].endswith(".json")]


frames = []
for key in keys:
    obj = s3.get_object(Bucket="health-data", Key=key)
    data = json.loads(obj["Body"].read())
    metric = next((m for m in data["data"]["metrics"] if m["name"] == "time_in_daylight"), None)
    if metric:
        df = pd.DataFrame(metric["data"])
        df["source_file"] = key
        frames.append(df)

# Union all into a single DuckDB query
con = duckdb.connect()
con.register("daylight", pd.concat(frames, ignore_index=True))
time_in_dl = con.execute("""
    SELECT date, qty AS time_in_daylight_min, source, source_file
    FROM daylight
    ORDER BY date
""").df()

time_in_dl['date'] = pd.to_datetime(time_in_dl['date'], utc=True).dt.tz_convert('America/New_York')
time_in_dl['day'] = pd.to_datetime(time_in_dl['date'].dt.strftime('%m/%d/%Y'))

time_in_dl.groupby('day').sum('time_in_daylight_min').sort_values('time_in_daylight_min', ascending = False)

,time_in_daylight_min
day,
2026-03-21,76.0
2026-03-19,63.0
2026-03-20,44.0
2026-03-18,42.0
2026-03-15,35.0
2026-03-17,7.0
